In [12]:
from utils.experiment_utils import get_all_experiments_info, load_best_model
import torch
import os
import hydra
from omegaconf import DictConfig, OmegaConf

import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

import ot

from datasets.lineage_tracing import LTSeqDataset

from geomloss import SamplesLoss

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

from utils.eval_utils import compute_mmd_distance, compute_sw_distance

from sklearn.neighbors import NearestNeighbors

In [13]:
lts = LTSeqDataset(seed=42)

loading cached adata from ./data/processed/adata_pca_50.h5ad  !!
loading cached clone sets from ./data/processed!
splitting 1218 clones into 609 train and 609 test


In [14]:
configs = get_all_experiments_info('/orcd/home/002/gokulg/orcd/scratch/CoupledDistributionEmbeddings/outputs/', False)

cfgs = [c for c in configs if 'lineage_semisupervised' in c['name'] and 'reg' not in c['name']
        and c['dataset'] != 'LTSeqDatasetUnstructuredBidirectional']

energy_models = [c for c in cfgs if 'mmd' in c['config']['generator'].values()
                 and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]
                #  and (c['config']['training']['num_epochs'] == 5000 or 'onehot' in c['name'])]
swd_models = [c for c in cfgs if 'swd' in c['config']['generator'].values()
              and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]
            #   and (c['config']['training']['num_epochs'] == 5000 or 'onehot' in c['name'])]
fm_models = [c for c in cfgs if 'Flow' in c['generator']
             and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]
            #  and (c['config']['training']['num_epochs'] == 5000 or 'onehot' in c['name'])]

reg_cfgs = [c for c in configs if 'lineage_semisupervised_direct_reg' in c['name'] 
        and c['dataset'] != 'LTSeqDatasetUnstructuredBidirectional']

energy_reg_models = [c for c in reg_cfgs if 'mmd' in c['config']['generator'].values()
                 and 'onehot' not in c['name'] and 'GNN' in c['config']['encoder']['_target_']]

# print("energy models:")
# for model in energy_models:
#     print(model['name'])

print('energy reg models:')
for model in energy_reg_models:
    print(model['name'])

# print("fm models:")
# for model in fm_models:
#     print(model['name'])

# print("swd models:")
# for model in swd_models:
#     print(model['name'])

energy reg models:
lineage_semisupervised_direct_reg_78c8cdfa412929893ae1204a36a849e1
lineage_semisupervised_direct_reg_a5e51ac852bbff5226eabdbea6cac8dc


In [15]:
def load_model(cfg, path, device):
    enc = hydra.utils.instantiate(cfg['encoder'])
    gen = hydra.utils.instantiate(cfg['generator'])
    state = load_best_model(path)
    enc.load_state_dict(state['encoder_state_dict'])
    gen.load_state_dict(state['generator_state_dict'])
    enc.eval()
    gen.eval()
    enc.to(device)
    gen.to(device)
    return enc, gen

In [16]:
results = {
    'generator' : [],
    'd_pair' : [],
    'd_rand' : []
}

energy = SamplesLoss('energy')

device = 'cuda'

# for model, name in zip(energy_models+energy_reg_models+fm_models+swd_models, ['energy', 'energy_reg', 'fm', 'swd']):
for model, name in zip(energy_reg_models, ['energy_reg_1', 'energy_reg_2']):
    print(f"Evaluating model: {model['name']}")
    encoder, generator = load_model(model['config'], model['dir'], 'cuda')
    

    with torch.no_grad():

        z_x_train = encoder(lts.train_srcs.to(device))
        z_y_train = encoder(lts.train_tgts.to(device))
        z_x_test = encoder(lts.test_srcs.to(device))
        z_y_test = encoder(lts.test_tgts.to(device))

    X_train = z_x_train.cpu().numpy()
    Y_train = z_y_train.cpu().numpy()
    X_test = z_x_test.cpu().numpy()
    Y_test = z_y_test.cpu().numpy()

    # fit ridge regression
    alphas = np.logspace(-6, 6, 25)
    ridge = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
    ridge.fit(X_train, Y_train)
    
    # predict target embeddings
    Y_pred_test = ridge.predict(X_test)
    mse = mean_squared_error(Y_test, Y_pred_test)
    r2 = r2_score(Y_test, Y_pred_test, multioutput='variance_weighted')
    
    # semi-supervised with predicted embeddings
    y_hat_semi = generator.sample(
        lts.test_srcs.to(device).reshape(-1, 50),
        z_x_test.to(device),
        torch.tensor(Y_pred_test, dtype=torch.float32).to(device)
    ).reshape(-1, 50)

    idx = torch.randperm(y_hat_semi.size(0))

    d_pair = (y_hat_semi - lts.test_srcs.reshape(-1, 50).to(device)).norm(dim=-1).mean().item()
    d_rand = (lts.test_srcs.reshape(-1, 50).to(device)[idx] - lts.test_tgts.reshape(-1, 50).to(device)).norm(dim=-1).mean().item()

    results['generator'].append(name)
    results['d_pair'].append(d_pair)
    results['d_rand'].append(d_rand)



Evaluating model: lineage_semisupervised_direct_reg_78c8cdfa412929893ae1204a36a849e1
Evaluating model: lineage_semisupervised_direct_reg_a5e51ac852bbff5226eabdbea6cac8dc


In [17]:
results_df = pd.DataFrame(results)
# results_df.groupby('generator').mean()
# ratio = d_pair / d_rand
print(results_df.groupby('generator').mean())
results_df.groupby('generator').mean()['d_pair'] / results_df.groupby('generator').mean()['d_rand']

                 d_pair     d_rand
generator                         
energy_reg_1  18.219458  26.978704
energy_reg_2  17.468582  26.970858


generator
energy_reg_1    0.675327
energy_reg_2    0.647684
dtype: float64

In [18]:
cell_type_results = {
    "generator": [],
    "clone" : [],
    "predicted target types" : [],
    "observed target types" : []
}

# construct nearest neighbor classifier for cell types

cell_type_classifier = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
cell_type_classifier.fit(lts.adata.obsm['X_pca'][:, :50])
cell_types = lts.adata.obs['type'].values

# for model, name in zip(energy_models+fm_models+swd_models, ['energy', 'fm', 'swd']):
for model, name in zip(energy_reg_models, ['energy_reg']):
    print(f"Evaluating model: {model['name']}")
    encoder, generator = load_model(model['config'], model['dir'], 'cuda')
    

    with torch.no_grad():

        z_x_train = encoder(lts.train_srcs.to(device))
        z_y_train = encoder(lts.train_tgts.to(device))
        z_x_test = encoder(lts.test_srcs.to(device))
        z_y_test = encoder(lts.test_tgts.to(device))

    X_train = z_x_train.cpu().numpy()
    Y_train = z_y_train.cpu().numpy()
    X_test = z_x_test.cpu().numpy()
    Y_test = z_y_test.cpu().numpy()

    # fit ridge regression
    alphas = np.logspace(-6, 6, 25)
    ridge = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
    ridge.fit(X_train, Y_train)
    
    # predict target embeddings
    Y_pred_test = ridge.predict(X_test)
    mse = mean_squared_error(Y_test, Y_pred_test)
    r2 = r2_score(Y_test, Y_pred_test, multioutput='variance_weighted')
    
    # semi-supervised with predicted embeddings
    y_hat_semi = generator.sample(
        lts.test_srcs.to(device).reshape(-1, 50),
        z_x_test.to(device),
        torch.tensor(Y_pred_test, dtype=torch.float32).to(device)
    ).reshape(lts.test_tgts.shape)

    # shape: clones, cells per clone, dims
    # predict cell type composition per clone for true targets and predicted targets

    for i in range(y_hat_semi.shape[0]):
        pred_target_embeddings = y_hat_semi[i].cpu().detach().numpy()
        _, pred_nn_idx = cell_type_classifier.kneighbors(pred_target_embeddings)
        pred_cell_types = cell_types[pred_nn_idx.flatten()]

        true_target_embeddings = lts.test_tgts[i].cpu().detach().numpy()
        _, true_nn_idx = cell_type_classifier.kneighbors(true_target_embeddings)
        true_cell_types = cell_types[true_nn_idx.flatten()]

        cell_type_results['generator'].append(name)
        cell_type_results['predicted target types'].append(pred_cell_types)
        cell_type_results['observed target types'].append(true_cell_types)
        cell_type_results['clone'].append(i)


Evaluating model: lineage_semisupervised_direct_reg_78c8cdfa412929893ae1204a36a849e1


In [19]:
cell_type_results_df = pd.DataFrame(cell_type_results)

# for each generator, compute MSE between predicted/observed cell type dists across clones

for generator in cell_type_results_df['generator'].unique():
    pred_types = cell_type_results_df[cell_type_results_df['generator'] == generator]['predicted target types']
    obs_types = cell_type_results_df[cell_type_results_df['generator'] == generator]['observed target types']
    
    mse_sum = 0
    for pred, obs in zip(pred_types, obs_types):
        pred_dist = pd.Series(pred).value_counts(normalize=True)
        obs_dist = pd.Series(obs).value_counts(normalize=True)
        mse_sum += mean_squared_error(pred_dist.reindex(obs_dist.index, fill_value=0).values, obs_dist.values)
    
    mse_avg = mse_sum / len(pred_types)
    print(f"Generator: {generator}, Average MSE between predicted and observed cell type distributions: {mse_avg}")

# for each generator compute MSE between predicted target types and 
# observed target types

# observed_target_distribution = lts.adata.obs['type'].value_counts(normalize=True)
# for generator in cell_type_results_df['generator'].unique():
#     pred_types = cell_type_results_df[cell_type_results_df['generator'] == generator]['predicted target type']
#     pred_distribution = pred_types.value_counts(normalize=True)
#     mse = mean_squared_error(observed_target_distribution.values, pred_distribution.reindex(observed_target_distribution.index, fill_value=0).values)
#     print(f"Generator: {generator}, MSE to observed distribution: {mse}")

Generator: energy_reg, Average MSE between predicted and observed cell type distributions: 0.0192812101910828
